# S42 답변 형식 통일: 근거 번호 문구를 더한 v16 결과

v15(문단)는 성능 기준을 통과했지만 인용 전무 답변이 늘었다. 문단을 나눠도 문장마다 근거 번호를 붙이라는 문구를 더한 v16을 gen-full로 재고, 기준 반영을 정할 근거를 낸다(`docs/slices/42-answer-format.md`, D-75 결정 6). 앞 결과는 `46_01`이다. 저장된 기록만 읽고 API를 부르지 않는다. 판정 전이므로 답변·질의 원문은 출력하지 않는다.

In [1]:
import json
import re
import statistics
import subprocess
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

from rag.gen_eval.checks import CITE, sentences

In [2]:
ROOT = Path.cwd().parent
STEM = "hybrid-cond-filter-law-ms256-k20-ce-{v}-dev-full-spk-marks{s}"
RUNS = ROOT / "data/gen_eval/runs"
COMPARE = {"v15": ROOT / "data/gen_eval/compare/s42-v15-vs-exp023.json", "v16": ROOT / "data/gen_eval/compare/s42-v16-vs-exp023.json"}
EXPS = {"v10": "exp023", "v15": "exp025", "v16": "exp026"}
REFUSAL_KEY = "찾지 못했습니다"
# 합의 기준(지시서 결정 사항 1): 안전 조건 세 가지와 종합 지표 구간 상한 ≥ 0
LIMITS = {"composite": ("상한0", 0.0), "faithfulness": ("하한", -0.05), "relevance": ("하한", -0.035), "over_refusal": ("상한", 0.02), "speaker_mismatch": ("상한", 0.02)}

In [3]:
BULLET = re.compile(r"^\s*[-*•]\s")


def load(version, suffix=""):
    path = RUNS / (STEM.format(v=version, s=suffix) + ".jsonl")
    with path.open(encoding="utf-8") as f:
        return {r["qid"]: r for r in map(json.loads, f)}


def shape(answer):
    lines = [x for x in answer.strip().split("\n") if x.strip()]
    n_bullet = sum(bool(BULLET.match(x)) for x in lines)
    if n_bullet == 0:
        return "한 문단" if len(lines) == 1 else "여러 문단"
    return "불릿만" if n_bullet == len(lines) else "불릿과 다른 줄"


def has_bullet(answer):
    return any(BULLET.match(x) for x in answer.split("\n"))


def passes(name, low, high):
    kind, limit = LIMITS[name]
    if kind == "하한":
        return low >= limit
    return high >= limit if kind == "상한0" else high <= limit

## 판정 지표

`rag.gen_eval.compare` 결과(회의 단위 paired bootstrap 95% 구간). 통과 칸은 지시서의 합의 기준이다. 포함도는 보조라 조건이 없다.

In [4]:
rows = []
for v, path in COMPARE.items():
    result = json.loads(path.read_text(encoding="utf-8"))
    for name, m in result["metrics"].items():
        low, high = m["ci"]
        passed = passes(name, low, high) if name in LIMITS else ""
        rows.append({"설정": v, "지표": name, "기준": round(m["base"], 3), "후보": round(m["cand"], 3), "차이": round(m["diff"], 4), "하한": round(low, 4), "상한": round(high, 4), "질의": m["n"], "통과": passed})
    print(v, "compare 판정", result["verdict"], result["failed"], "judge 설정 차이", result["check_settings_diff"], "충실성 형식 오류 후보", result["exclusions"]["cand"]["faithfulness"]["format_errors"], "기준", result["exclusions"]["base"]["faithfulness"]["format_errors"])
pd.DataFrame(rows)

v15 compare 판정 보류 ['종합 지표 구간 하한 ≤ 0', '종합 지표 점 추정 < 0.02'] judge 설정 차이 {} 충실성 형식 오류 후보 5 기준 4
v16 compare 판정 보류 ['종합 지표 구간 하한 ≤ 0', '종합 지표 점 추정 < 0.02'] judge 설정 차이 {} 충실성 형식 오류 후보 4 기준 4


,설정,지표,기준,후보,차이,하한,상한,질의,통과
0,v15,faithfulness,0.983,0.982,-0.0002,-0.0108,0.0117,318,True
1,v15,relevance,0.882,0.886,0.0046,-0.0175,0.0269,325,True
2,v15,composite,0.933,0.934,0.0007,-0.0119,0.0134,318,True
3,v15,inclusion,0.709,0.727,0.0180,-0.0094,0.0449,265,
4,v15,over_refusal,0.010,0.000,-0.0105,-0.0210,-0.0018,286,True
5,v15,speaker_mismatch,0.009,0.013,0.0044,-0.0050,0.0125,318,True
6,v16,faithfulness,0.983,0.991,0.0087,0.0008,0.0186,319,True
7,v16,relevance,0.882,0.878,-0.0031,-0.0250,0.0189,325,True
8,v16,composite,0.933,0.935,0.0028,-0.0091,0.0146,319,True
9,v16,inclusion,0.709,0.712,0.0031,-0.0226,0.0290,265,


## 형식

답변 모양과 재생성 흔들림. 비교값은 v2·v4 재실행의 2\~3%(D-75 이유)이고, v10은 재생성하지 않았다.

In [5]:
runs = {"v10": load("v10"), "v15": load("v15"), "v16": load("v16")}
reruns = {"v15": load("v15", "-r1"), "v16": load("v16", "-r1")}
table = {v: Counter(shape(r["answer"]) for r in d.values()) for v, d in runs.items()}
pd.DataFrame(table).fillna(0).astype(int)

,v10,v15,v16
여러 문단,62,142,138
불릿과 다른 줄,37,0,0
한 문단,224,188,192
불릿만,7,0,0


In [6]:
rows = []
for v, d in runs.items():
    lengths = [len(r["answer"]) for r in d.values()]
    changed = sum(has_bullet(runs["v10"][q]["answer"]) != has_bullet(r["answer"]) for q, r in d.items())
    flip = sum(has_bullet(r["answer"]) != has_bullet(reruns[v][q]["answer"]) for q, r in d.items()) if v in reruns else None
    rows.append({"설정": v, "글자 중앙값": int(statistics.median(lengths)), "글자 평균": int(statistics.mean(lengths)), "v10과 불릿 유무 다름": changed, "재생성 불릿 유무 다름": flip})
pd.DataFrame(rows)

,설정,글자 중앙값,글자 평균,v10과 불릿 유무 다름,재생성 불릿 유무 다름
0,v10,255,299,0,NaN
1,v15,279,325,44,1.0
2,v16,274,324,44,0.0


## 인용

거부 문장을 뺀 문장 기준이다. 인용이 전혀 없는 답변은 D-76에서 v10을 채택한 이유 중 하나다(v9 11 → v10 1).

In [7]:
rows = []
named = {**runs, "v15 재생성": reruns["v15"], "v16 재생성": reruns["v16"]}
for v, d in named.items():
    no_cite = uncited = total = 0
    for r in d.values():
        body = [s for s in sentences(r["answer"]) if REFUSAL_KEY not in s]
        no_cite += bool(body) and not CITE.search(r["answer"])
        uncited += sum(not CITE.search(s) for s in body)
        total += len(body)
    rows.append({"설정": v, "인용 전무 답변": no_cite, "인용 없는 문장": uncited, "문장": total, "비율": round(uncited / total, 3), "문장/답변": round(total / len(d), 2)})
pd.DataFrame(rows)

,설정,인용 전무 답변,인용 없는 문장,문장,비율,문장/답변
0,v10,1,55,795,0.069,2.41
1,v15,18,62,853,0.073,2.58
2,v16,5,19,860,0.022,2.61
3,v15 재생성,24,90,868,0.104,2.63
4,v16 재생성,8,35,869,0.040,2.63


## report 관찰

거부, 항목 단위 처리, 지연·비용.

In [8]:
keep = re.compile(r"전체 거부|변형 거부, |틀린 처리가|^\| (근거 없이|안내 없이|무언|누락|과잉 거부|불필요) |인용 없는|내용 없는|불일치가 있는 답변|생성 비용|첫 토큰|전체 시간|끝난 이유")
for v, exp in EXPS.items():
    out = subprocess.run([sys.executable, "-m", "rag.gen_eval.report", f"+exp={exp}"], cwd=ROOT, capture_output=True, encoding="utf-8", check=True).stdout
    print("===", v)
    print("\n".join(x[:110] for x in out.splitlines() if keep.search(x)))

=== v10
질의 330개, 전체 거부 5/330 (1.5%), 부분 거부 23/330 (7.0%)
| 변형 거부, 전체 거부로 셈 | 2 |
| 변형 거부, 부분 거부로 셈 | 7 |
| 변형 거부, 거부로 세지 않음(S34 전 judge 결과) | 0 |
질의 항목 단위 거부 평가(S14c): 질의 330개, 판단 불가(judge 형식 오류) 0개, 틀린 처리가 있는 질의 115/330 (34.8%)
| 근거 없이 답함 | 53 |
| 안내 없이 부분 답함 | 50 |
| 무언 생략 | 26 |
| 누락 | 15 |
| 과잉 거부 | 5 |
| 인용 없는 문장 | 41/780 (5.3%) |
| 내용 없는 문장(인사말·도입 문장, 문장 수에서 뺌) | 15 |
| 회의 일자·위원회 불일치가 있는 답변 | 0/325 (0.0%) |
| 생성 비용 추정 | $0.1230 |
| 답변 첫 토큰 시간 중앙값·p95·최대 | 0.72초 · 3.62초 · 10.96초 |
| 생성 전체 시간 중앙값·p95 | 3.73초 · 13.46초 |
| 생성 끝난 이유 | stop 330 |
=== v15
질의 330개, 전체 거부 3/330 (0.9%), 부분 거부 14/330 (4.2%)
| 변형 거부, 전체 거부로 셈 | 1 |
| 변형 거부, 부분 거부로 셈 | 2 |
| 변형 거부, 거부로 세지 않음(S34 전 judge 결과) | 0 |
질의 항목 단위 거부 평가(S14c): 질의 330개, 판단 불가(judge 형식 오류) 0개, 틀린 처리가 있는 질의 116/330 (35.2%)
| 근거 없이 답함 | 61 |
| 안내 없이 부분 답함 | 55 |
| 무언 생략 | 26 |
| 누락 | 12 |
| 인용 없는 문장 | 61/852 (7.2%) |
| 내용 없는 문장(인사말·도입 문장, 문장 수에서 뺌) | 1 |
| 회의 일자·위원회 불일치가 있는 답변 | 0/327 (0.0%) |
| 생성 비용 추정 | $0.1259 |
| 답변 첫 토큰 시간 중앙값·p95·최